# FreshGuard â€” Notebook 2 / 5: Pseudoâ€‘Label Detection Subset with Grounding DINO

**Purpose.** Generate YOLOâ€‘format bounding boxes for a stratified subset of the deduplicated manifest, using the **known** `(produce_type, freshness)` label from the manifest as the YOLO class and using **Grounding DINO** only to localize. We're not asking DINO to classify â€” we already know what's in the image.

**Why this matters.** The previous detector was trained on only 390 images. Recall was 0.438 (â‰ˆ56% miss rate). This notebook produces â‰¥5,000 boxes, with a perâ€‘class floor of 300 to give minority classes (Bittergroud, Okara, Mango, Strawberry) a fighting chance.

**Inputs (attach as Kaggle Datasets):**
- `ulnnproject/food-freshness-dataset` â€” the raw images
- `<your-handle>/freshguard-manifest` â€” `manifest.parquet` from notebook 1

**Outputs (publish as `<your-handle>/freshguard-detector-data`):**
- `images/{train,val,test}/` â€” symlinks or copies of the sampled images
- `labels/{train,val,test}/` â€” YOLO `.txt` label files
- `detector_dataset.yaml` â€” Ultralytics dataset spec (24 classes)
- `qa_report.md` â€” perâ€‘class accept/reject counts and a 200â€‘image QA contact sheet

**Expected runtime.** â‰ˆ45â€“90 minutes on a Kaggle T4 GPU (DINO is the bottleneck).

**GPU required.** Set the notebook's accelerator to GPU T4 Ã—2 or P100 in the rightâ€‘hand sidebar.

In [ ]:
!pip install --quiet "transformers>=4.41,<5" pyarrow

In [ ]:
from __future__ import annotations

import json
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = False

# ---------------------------------------------------------------
# Configuration â€” edit paths and perâ€‘class targets if needed.
# ---------------------------------------------------------------
MANIFEST_PATH = Path("/kaggle/input/datasets/elsmmany/freshguard-manifest/manifest.parquet")
OUTPUT_DIR = Path("/kaggle/working/detector")
DINO_MODEL_ID = "IDEA-Research/grounding-dino-tiny"

# Perâ€‘class image cap. Minorities get a higher floor so the detector has a
# fighting chance on Bittergroud/Okara/Mango/Strawberry; majorities are capped
# so we don't waste compute on Tomato/Orange/Carrot which are already abundant.
PER_CLASS_TARGET = 600  # default
PER_CLASS_FLOOR = 300   # never sample fewer than this if available
PER_CLASS_CAP = 800     # never sample more than this even if available

BOX_CONFIDENCE = 0.30        # DINO score threshold
BOX_TEXT_THRESHOLD = 0.25
MIN_BOX_AREA_FRAC = 0.05     # reject tiny boxes (likely shadows/labels)
MAX_BOX_AREA_FRAC = 0.95     # reject fullâ€‘frame boxes (likely background grab)
MAX_BOXES_PER_IMAGE = 6

QA_PER_CLASS_SAMPLES = 8     # 8 Ã— 24 â‰ˆ 192 images for the QA contact sheet

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for split in ("train", "val", "test"):
    (OUTPUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "qa").mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    print("WARNING: no GPU detected. Grounding DINO will be very slow on CPU.")

## Class names + DINO prompts

DINO needs naturalâ€‘language prompts. We map our 12 canonical produce types to short, unambiguous noun prompts. Some classes (Bell Pepper, Bitter Gourd) get alias prompts to widen DINO's recall.

In [ ]:
PRODUCE_TYPES = (
    "apple", "banana", "bellpepper", "bitter_gourd", "carrot", "cucumber",
    "mango", "okra", "orange", "potato", "strawberry", "tomato",
)
FRESHNESS_LEVELS = ("fresh", "rotten")
COMBINED_CLASSES = tuple(f"{t}_{f}" for t in PRODUCE_TYPES for f in FRESHNESS_LEVELS)
CLASS_TO_INDEX = {c: i for i, c in enumerate(COMBINED_CLASSES)}
assert len(COMBINED_CLASSES) == 24

# DINO needs natural-language prompts. We accept the most likely synonyms.
DINO_PROMPTS: dict[str, list[str]] = {
    "apple": ["apple"],
    "banana": ["banana"],
    "bellpepper": ["bell pepper", "capsicum", "sweet pepper"],
    "bitter_gourd": ["bitter gourd", "bitter melon"],
    "carrot": ["carrot"],
    "cucumber": ["cucumber"],
    "mango": ["mango"],
    "okra": ["okra", "lady finger"],
    "orange": ["orange"],
    "potato": ["potato"],
    "strawberry": ["strawberry"],
    "tomato": ["tomato"],
}

## Step 1 â€” Sample a stratified subset from the manifest

For each `(produce_type, freshness, split)` group we draw up to `PER_CLASS_TARGET` images, with a `PER_CLASS_FLOOR` to protect minorities and a `PER_CLASS_CAP` to keep DINO compute bounded. Sampling respects the splits frozen in notebook 1, so train/val/test stay disjoint here too.

In [ ]:
manifest = pd.read_parquet(MANIFEST_PATH)
print(f"Loaded manifest: {len(manifest):,} rows, {manifest.combined_label.nunique()} classes.")

# Translate paths from older notebook-1 runs that used the bare /kaggle/input/<slug>/
# prefix instead of the user's /kaggle/input/datasets/<owner>/<slug>/ mount pattern.
OLD_PREFIX = "/kaggle/input/food-freshness-dataset/"
NEW_PREFIX = "/kaggle/input/datasets/ulnnproject/food-freshness-dataset/"
n_translated = int(manifest["path"].str.startswith(OLD_PREFIX).sum())
if n_translated:
    manifest["path"] = manifest["path"].str.replace(OLD_PREFIX, NEW_PREFIX, regex=False)
    print(f"Translated {n_translated:,} legacy paths to the datasets/<owner>/ mount.")

# Fail loudly if paths don't resolve, instead of silently skipping in the DINO loop.
sample_paths = manifest["path"].sample(min(10, len(manifest)), random_state=0).tolist()
n_exist = sum(1 for p in sample_paths if Path(p).exists())
print(f"Path resolution sanity: {n_exist}/{len(sample_paths)} sampled paths exist on disk.")
if n_exist == 0:
    raise FileNotFoundError(
        "None of the sampled manifest paths exist. The dataset is probably mounted "
        "at a different path. Open the right-sidebar input panel on Kaggle, copy the "
        "actual mount path of food-freshness-dataset, and edit MANIFEST_PATH and the "
        "OLD_PREFIX / NEW_PREFIX constants above to match."
    )

rng = np.random.default_rng(0)
samples: list[pd.DataFrame] = []
for (label, split), group in manifest.groupby(["combined_label", "split"]):
    n_available = len(group)
    target = min(max(PER_CLASS_FLOOR, PER_CLASS_TARGET), PER_CLASS_CAP, n_available)
    # Per-split share is proportional to the global TRAIN/VAL/TEST ratio (70/15/15).
    split_frac = {"train": 0.70, "val": 0.15, "test": 0.15}[split]
    sample_n = max(1, int(round(target * split_frac))) if split != "train" else target
    sample_n = min(sample_n, n_available)
    chosen = group.sample(n=sample_n, random_state=int(rng.integers(0, 2**31)))
    samples.append(chosen)

subset = pd.concat(samples, ignore_index=True)
print(f"Sampled {len(subset):,} images across {subset.combined_label.nunique()} classes.")
display(subset.groupby(["combined_label", "split"]).size().unstack(fill_value=0))

## Step 2 â€” Load Grounding DINO and define the box generator

We use `IDEA-Research/grounding-dino-tiny` from Hugging Face â€” small enough to run quickly on a Kaggle T4 GPU but accurate enough for produce. For each image we run DINO with the produceâ€‘type's prompts (joined with `. `, the recommended Grounding DINO prompt format) and accept boxes that pass the area filter.

In [ ]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
import inspect

DINO_BATCH_SIZE = 2  # images per forward pass per GPU — T4 (16 GiB) tolerates 2 safely; bump to 4 if you have headroom

processor = AutoProcessor.from_pretrained(DINO_MODEL_ID)
dino_base = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_MODEL_ID).to(device).eval()
n_gpus = torch.cuda.device_count()
if n_gpus > 1:
    print(f"Using DataParallel across {n_gpus} GPUs.")
    dino_model = torch.nn.DataParallel(dino_base, device_ids=list(range(n_gpus)))
    EFFECTIVE_BATCH = DINO_BATCH_SIZE * n_gpus
else:
    dino_model = dino_base
    EFFECTIVE_BATCH = DINO_BATCH_SIZE

# transformers >=4.46 renamed `box_threshold` to `threshold`.
_post = processor.post_process_grounded_object_detection
_BOX_KW = "threshold" if "threshold" in inspect.signature(_post).parameters else "box_threshold"
print(f"Grounding DINO post-process kwarg: {_BOX_KW} | per-batch images: {EFFECTIVE_BATCH}")


@torch.no_grad()
def detect_boxes_batch(images, produce_types):
    """Run DINO on a batch. Returns list of [(x1,y1,x2,y2), ...] per input image."""
    texts = [". ".join(DINO_PROMPTS[pt]) + "." for pt in produce_types]
    inputs = processor(images=images, text=texts, return_tensors="pt", padding=True).to(device)
    outputs = dino_model(**inputs)
    target_sizes = torch.tensor([img.size[::-1] for img in images], device=device)
    results = _post(
        outputs,
        inputs.input_ids,
        text_threshold=BOX_TEXT_THRESHOLD,
        target_sizes=target_sizes,
        **{_BOX_KW: BOX_CONFIDENCE},
    )
    out = []
    for r in results:
        boxes = r["boxes"].cpu().numpy()
        scores = r["scores"].cpu().numpy()
        if len(boxes) == 0:
            out.append([])
            continue
        order = np.argsort(-scores)
        boxes = boxes[order][:MAX_BOXES_PER_IMAGE]
        out.append([tuple(map(float, b)) for b in boxes])
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out


@torch.no_grad()
def detect_boxes(image, produce_type):
    """Single-image wrapper (kept for back-compat)."""
    return detect_boxes_batch([image], [produce_type])[0]


## Step 3 â€” Run DINO on every sampled image, write YOLO labels

YOLO format: one `.txt` per image, one line per box: `class_id cx cy w h` (all coords normalized to [0, 1]). The class id corresponds to the 24â€‘class `COMBINED_CLASSES` ordering.

In [ ]:
def yolo_line(class_id, box, img_w, img_h):
    x1, y1, x2, y2 = box
    bw = (x2 - x1) / img_w
    bh = (y2 - y1) / img_h
    area_frac = bw * bh
    if area_frac < MIN_BOX_AREA_FRAC or area_frac > MAX_BOX_AREA_FRAC:
        return None
    cx = (x1 + x2) / 2 / img_w
    cy = (y1 + y2) / 2 / img_h
    if not (0 < cx < 1 and 0 < cy < 1 and 0 < bw < 1 and 0 < bh < 1):
        return None
    return f"{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"


stats_total = Counter()
stats_kept_boxes = Counter()
stats_no_box = Counter()
rejected_area = Counter()
manifest_rows = []


def _flush_batch(batch_meta, batch_images, batch_types):
    """Run DINO on the buffered batch and write YOLO labels for each image."""
    if not batch_meta:
        return
    boxes_per_image = detect_boxes_batch(batch_images, batch_types)
    for (src_path, row), image, boxes in zip(batch_meta, batch_images, boxes_per_image):
        img_w, img_h = image.size
        class_id = CLASS_TO_INDEX[row.combined_label]
        label_lines = []
        for box in boxes:
            line = yolo_line(class_id, box, img_w, img_h)
            if line is None:
                rejected_area[row.combined_label] += 1
                continue
            label_lines.append(line)
        stats_total[row.combined_label] += 1
        if not label_lines:
            stats_no_box[row.combined_label] += 1
            continue
        stats_kept_boxes[row.combined_label] += len(label_lines)
        out_image = OUTPUT_DIR / "images" / row.split / src_path.name
        out_label = OUTPUT_DIR / "labels" / row.split / (src_path.stem + ".txt")
        if not out_image.exists():
            # Always copy. Kaggle Dataset commit drops external
            # symlinks, which would publish an empty images/ dir.
            shutil.copy2(src_path, out_image)
        out_label.write_text("\n".join(label_lines) + "\n")
        manifest_rows.append({
            "image": str(out_image),
            "label": str(out_label),
            "produce_type": row.produce_type,
            "freshness": row.freshness,
            "combined_label": row.combined_label,
            "split": row.split,
            "n_boxes": len(label_lines),
        })


batch_meta, batch_images, batch_types = [], [], []
for idx, row in enumerate(subset.itertuples(index=False)):
    src_path = Path(row.path)
    if not src_path.exists():
        continue
    try:
        with Image.open(src_path) as img:
            image = img.convert("RGB")
    except Exception:
        continue
    batch_meta.append((src_path, row))
    batch_images.append(image)
    batch_types.append(row.produce_type)
    if len(batch_meta) >= EFFECTIVE_BATCH:
        _flush_batch(batch_meta, batch_images, batch_types)
        batch_meta, batch_images, batch_types = [], [], []
        if (idx + 1) % 200 == 0:
            kept_so_far = sum(stats_kept_boxes.values())
            print(f"  processed {idx + 1:,} / {len(subset):,} | boxes kept so far: {kept_so_far:,}")

_flush_batch(batch_meta, batch_images, batch_types)

label_manifest = pd.DataFrame(manifest_rows)
print(f"\nDone. {len(label_manifest):,} images received at least one accepted box.")
print(f"Total accepted boxes: {sum(stats_kept_boxes.values()):,}")


## Step 4 â€” Perâ€‘class accept/reject summary

Watch the minority classes carefully. If a minority class has too many `no_box` rows, lower `BOX_CONFIDENCE` for that class or add more prompt aliases. Document the outcome in the QA report â€” we're being honest about pseudoâ€‘label noise.

In [ ]:
summary = pd.DataFrame(
    {
        "sampled": [int(stats_total.get(c, 0)) for c in COMBINED_CLASSES],
        "no_box": [int(stats_no_box.get(c, 0)) for c in COMBINED_CLASSES],
        "rejected_area": [int(rejected_area.get(c, 0)) for c in COMBINED_CLASSES],
        "boxes_kept": [int(stats_kept_boxes.get(c, 0)) for c in COMBINED_CLASSES],
    },
    index=list(COMBINED_CLASSES),
)
summary["images_with_boxes"] = summary.sampled - summary.no_box
denom = summary.sampled.where(summary.sampled > 0)
summary["box_recall"] = (summary.images_with_boxes / denom).round(3).fillna(0.0)
summary = summary[["sampled", "images_with_boxes", "no_box", "rejected_area", "boxes_kept", "box_recall"]]
display(summary)

minority_classes = [c for c in COMBINED_CLASSES if c.startswith(("bitter_gourd", "okra", "mango", "strawberry"))]
print("Minority class box recall:")
display(summary.loc[minority_classes, ["sampled", "images_with_boxes", "box_recall"]])

## Step 5 â€” QA contact sheet

Sample 8 images per class with their predicted boxes drawn, then write a single grid image. Open the contact sheet and eyeball it. Record the visual noise rate per class in the report below.

In [ ]:
if label_manifest.empty:
    print("label_manifest is empty — skipping QA contact sheet.")
    print("Likely cause: the DINO loop processed zero images. Re-run the diagnostic")
    print("cell above; if all classes show processed=0, the path translation in")
    print("the manifest-load cell didn't catch your prefix.")
    qa_path = None
else:
    def draw_boxes_on(image: Image.Image, label_path: Path, color: str = "red") -> Image.Image:
        image = image.copy()
        draw = ImageDraw.Draw(image)
        w, h = image.size
        for line in label_path.read_text().strip().splitlines():
            _cls, cx, cy, bw, bh = line.split()
            cx, cy, bw, bh = (float(v) for v in (cx, cy, bw, bh))
            x1 = (cx - bw / 2) * w
            y1 = (cy - bh / 2) * h
            x2 = (cx + bw / 2) * w
            y2 = (cy + bh / 2) * h
            draw.rectangle((x1, y1, x2, y2), outline=color, width=4)
        return image

    qa_sample = (
        label_manifest
        .groupby("combined_label", group_keys=False)
        .apply(lambda g: g.sample(n=min(QA_PER_CLASS_SAMPLES, len(g)), random_state=0))
    )
    tile_w = tile_h = 256
    n_cols = QA_PER_CLASS_SAMPLES
    n_rows = len(COMBINED_CLASSES)
    sheet = Image.new("RGB", (tile_w * n_cols, tile_h * n_rows), color=(20, 20, 20))
    draw = ImageDraw.Draw(sheet)
    for row_idx, label in enumerate(COMBINED_CLASSES):
        rows = qa_sample[qa_sample.combined_label == label].head(QA_PER_CLASS_SAMPLES)
        for col_idx, qa_row in enumerate(rows.itertuples(index=False)):
            try:
                with Image.open(qa_row.image) as img:
                    annotated = draw_boxes_on(img.convert("RGB"), Path(qa_row.label))
            except Exception:
                continue
            annotated.thumbnail((tile_w, tile_h))
            sheet.paste(annotated, (col_idx * tile_w, row_idx * tile_h))
        draw.text((4, row_idx * tile_h + 4), label, fill="white")
    qa_path = OUTPUT_DIR / "qa" / "contact_sheet.jpg"
    sheet.save(qa_path, quality=85)
    print(f"QA contact sheet: {qa_path}")
    from IPython.display import Image as IPyImage
    IPyImage(filename=str(qa_path))

## Step 6 â€” Write the Ultralytics dataset YAML and the QA report

`detector_dataset.yaml` is the file you'll pass to `YOLO.train(data=...)` in notebook 3.

In [ ]:
yaml_text = (
    f"# Ultralytics YOLO dataset spec â€” produced by notebook 2.\n"
    f"path: {OUTPUT_DIR}\n"
    f"train: images/train\n"
    f"val: images/val\n"
    f"test: images/test\n"
    f"nc: {len(COMBINED_CLASSES)}\n"
    f"names:\n"
)
yaml_text += "".join(f"  {i}: {name}\n" for i, name in enumerate(COMBINED_CLASSES))
(OUTPUT_DIR / "detector_dataset.yaml").write_text(yaml_text)
print("Wrote detector_dataset.yaml:")
print(yaml_text)

report_lines = [
    "# Pseudo-Label QA Report (notebook 2)",
    "",
    f"- Sampled images: **{len(subset):,}**",
    f"- Images with at least one accepted box: **{len(label_manifest):,}**",
    f"- Total accepted boxes: **{int(summary.boxes_kept.sum()):,}**",
    f"- DINO model: `{DINO_MODEL_ID}`",
    f"- Box conf threshold: {BOX_CONFIDENCE} | text threshold: {BOX_TEXT_THRESHOLD}",
    f"- Area filter: keep boxes between {MIN_BOX_AREA_FRAC:.0%} and {MAX_BOX_AREA_FRAC:.0%} of image area",
    "",
    "## Perâ€‘class box recall",
    "",
    summary.to_markdown(),
    "",
    "## Manual QA",
    "",
    f"Open `{qa_path.relative_to(OUTPUT_DIR)}` and eyeball each row. Record perâ€‘class noise rate below before running notebook 3.",
    "",
    "| Class | Sampled in QA | Wrong/Empty boxes | Noise % |",
    "|---|---:|---:|---:|",
    *[f"| {c} | {QA_PER_CLASS_SAMPLES} | _fill in_ | _fill in_ |" for c in COMBINED_CLASSES],
]
(OUTPUT_DIR / "qa_report.md").write_text("\n".join(report_lines))
print(f"Wrote: {OUTPUT_DIR / 'qa_report.md'}")

## Publish as a Kaggle Dataset

1. **Save Version â†’ Save & Run All (Commit)**.
2. From the notebook **Output** tab, **New Dataset** â†’ name it `freshguard-detector-data`.
3. Notebook 3 attaches this dataset and trains YOLO26s on it.

**Verification before publishing:**
- Every class in `summary` has `images_with_boxes â‰¥ 200` (raise `PER_CLASS_TARGET` and reâ€‘run if any class is below).
- The contact sheet looks reasonable for at least the four minority classes (Bittergroud, Okra, Mango, Strawberry).
- `detector_dataset.yaml` lists all 24 classes in the canonical order.